# NCAERS — Ablation study: ContextEncoder vs BodyEncoder vs complete model

| Model | Input | Pipe |
|--------|---------|-------------|
| **ContextOnly** | full image | ResNet50 Places365 → Linear(256,7) |
| **BodyOnly** | face crop (Haar cascade) | ResNet50 ImageNet → Linear(256,7) |
| **CCIM completo** | full image + face crop | Both encoders + merge + CCIM |

## 1. Imports & Config

In [ ]:
from pathlib import Path

from dual_resnet_shared import (
    device, NCAERS_LABELS,
    _face_cascade,
    make_ncaers_loaders,
)
from ablation_models import (
    ContextOnlyModel, BodyOnlyModel,
    train_ablation_model, evaluate_ablation_model,
)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

CKPT_DIR = Path("checkpoints")
ABLATION_CKPT_DIR = CKPT_DIR / "ablation"
ABLATION_CKPT_DIR.mkdir(parents=True, exist_ok=True)

FACE_CROPS_ROOT = Path("datasets/NCAERS_faces")
FACE_CROPS_INDEX = FACE_CROPS_ROOT / "ncaers_faces_index.parquet"
FACE_CROPS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Ablation checkpoints: {ABLATION_CKPT_DIR.resolve()}")

## 2. Load data

In [ ]:
train_loader, val_loader, test_loader, class_weights, _ = make_ncaers_loaders()
print(f"Loaders: train={len(train_loader.dataset)} | val={len(val_loader.dataset)} | test={len(test_loader.dataset)}")

## 3. Make and save face crops

In [ ]:
def detect_face_crop_with_info(img: Image.Image, padding: float = 0.15):
    W, H  = img.size
    gray  = np.array(img.convert("L"))
    faces = _face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(20, 20))
    if not len(faces):
        return img, False, (0, 0, W, H)
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x, pad_y = int(w * padding), int(h * padding)
    x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
    x2, y2 = min(W, x + w + pad_x), min(H, y + h + pad_y)
    return img.crop((x1, y1, x2, y2)), True, (x1, y1, x2, y2)

In [ ]:
def save_ncaers_face_crops(
    ncaers_root,
    output_root,
    padding = 0.15,
    overwrite = False,
):
    # Avoid re-executing
    if not overwrite and FACE_CROPS_INDEX.exists():
        df  = pd.read_parquet(FACE_CROPS_INDEX)
        pct = df["face_detected"].mean() * 100
        print(f"Index loaded: {len(df)} images ({pct:.1f}% with face detected.)")
        return df

    records = []
    for split_dir in sorted(d for d in ncaers_root.iterdir() if d.is_dir()):
        split = split_dir.name
        for label_dir in sorted(split_dir.iterdir()):
            if not label_dir.is_dir():
                continue
            label = label_dir.name
            out_dir = output_root / split / label
            out_dir.mkdir(parents=True, exist_ok=True)
            img_files = sorted(label_dir.glob("*.jpg"))
            if not img_files:
                img_files = sorted(label_dir.glob("*.png"))
            for img_file in tqdm(img_files, desc=f"{split}/{label}", leave=False):
                img = Image.open(img_file).convert("RGB")
                orig_w, orig_h = img.size
                crop, detected, (x1, y1, x2, y2) = detect_face_crop_with_info(img, padding)
                out_path = out_dir / img_file.name
                crop.save(out_path, "JPEG", quality=95)
                records.append({
                    "split": split, "label": label, "filename": img_file.name,
                    "face_detected": detected,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "orig_w": orig_w, "orig_h": orig_h,
                    "crop_w": crop.width, "crop_h": crop.height,
                })

    df = pd.DataFrame(records)
    df.to_parquet(FACE_CROPS_INDEX, index=False)
    pct = df["face_detected"].mean() * 100
    print(f"{len(df)} crops saved in {output_root}")
    print(f"Face detected: {pct:.1f}% | Fallback: {100 - pct:.1f}%")
    return df

In [ ]:
face_crops_df = save_ncaers_face_crops(overwrite=False)

## 4. ContextOnly

### Training

In [ ]:
ctx_model = ContextOnlyModel(n_classes=len(NCAERS_LABELS), freeze_backbone=True).to(device)

# Avoid re-execution
if (ABLATION_CKPT_DIR / "ncaers_context_only_best.pt").exists():
    ctx_model.load_state_dict(torch.load(
        ABLATION_CKPT_DIR / "ncaers_context_only_best.pt",
        map_location=device, weights_only=True,
    ))
    print(f"ContextOnly loaded from {ABLATION_CKPT_DIR / "ncaers_context_only_best.pt"}")
else:
    ctx_history = train_ablation_model(
        ctx_model, train_loader, val_loader, class_weights,
        tag="ContextOnly",
        ckpt_path=ABLATION_CKPT_DIR / "ncaers_context_only_best.pt",
    )

### Evaluation

In [ ]:
ctx_metrics = evaluate_ablation_model(
    ctx_model, test_loader, NCAERS_LABELS,
    tag="ContextOnly",
    save_prefix=ABLATION_CKPT_DIR / "ncaers_context_only",
)

## 5. BodyOnly

### Training

In [ ]:
body_model = BodyOnlyModel(n_classes=len(NCAERS_LABELS), freeze_backbone=True).to(device)

if (ABLATION_CKPT_DIR / "ncaers_body_only_best.pt").exists():
    body_model.load_state_dict(torch.load(
        ABLATION_CKPT_DIR / "ncaers_body_only_best.pt",
        map_location=device, weights_only=True,
    ))
    print(f"BodyOnly loaded from {ABLATION_CKPT_DIR / "ncaers_body_only_best.pt"}")
else:
    body_history = train_ablation_model(
        body_model, train_loader, val_loader, class_weights,
        tag="BodyOnly",
        ckpt_path=ABLATION_CKPT_DIR / "ncaers_body_only_best.pt",
    )

### Evaluation

In [ ]:
body_metrics = evaluate_ablation_model(
    body_model, test_loader, NCAERS_LABELS,
    tag="BodyOnly",
    save_prefix=ABLATION_CKPT_DIR / "ncaers_body_only",
)